In [1]:
# # Gather links with unstructured
# from unstructured.partition.html import partition_html
# cnn_lite_url = "https://lite.cnn.com/"
# elements = partition_html(url=cnn_lite_url)

# links = []

# for element in elements:
#     if element.metadata.link_urls:
#         relative_link = element.metadata.link_urls[0][1:]
#         if relative_link.startswith("2024"):
#             links.append(f"{cnn_lite_url}{relative_link}")

# print(f"We retrieved {len(links)} links to documents from {cnn_lite_url}")

In [2]:
# # Ingest individual articles with Langchain UnstructuredURLLoader
# from langchain.document_loaders import UnstructuredURLLoader
# loaders = UnstructuredURLLoader(urls=links[:200], show_progress_bar=True)

# docs = loaders.load()
# #print(docs[0])
# print(f"We retrieved {len(docs)} links to documents from {cnn_lite_url}")

# # Add id to each document to retrieve it if needed
# for i in range(len(docs)):
#     docs[i].id = i
#     docs[i].metadata['rewritten'] = False

# # Variable docs is a list of langchain_core.documents.base.Document -> convert it to a list of strings
# #docs_as_list_of_strings = [docs[i].page_content for i in range(len(docs))]

In [3]:
# For reproducibility we can save and load the documents in a file
# -------------------------------------------------------
# Save the list of documents as a JSON file
# import json
# with open("docs_as_list_of_strings.json", "w") as file:
#     json.dump(docs_as_list_of_strings, file)

# Load the list of documents from a JSON file
import json
with open("docs_as_list_of_strings.json", "r") as file:
    docs_as_list_of_strings = json.load(file)

# We work on documents, so convert the list of string into a list of documents
from langchain.docstore.document import Document
docs = []
i=0
for text in docs_as_list_of_strings:
    docs.append(Document(
        page_content=text,
        metadata={"source": "local", "rewritten": False},
        id=i
    ))
    i+=1
# -------------------------------------------------------

# Each document of the list is a very long text, so we split each document into smaller chuncks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=100,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)

# Two-dimensional list where each sublist represent a document splitted in chunks
docs_as_list_of_chuncks = [] 
for doc in docs:
    docs_as_list_of_chuncks.append(text_splitter.split_text(doc.page_content))

# Load ibm-granite/granite-guardian-hap-38m model
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
model_name_or_path = 'ibm-granite/granite-guardian-hap-38m'
model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path)
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

# Calculate HAP probability for every chunk. Save predictions and probabilities for every chunk in a two-dimensional list
prediction_results = []
probability_results = []
for chunks in docs_as_list_of_chuncks:
    input = tokenizer(chunks, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        logits = model(**input).logits
        prediction_list = torch.argmax(logits, dim=1).detach().numpy().tolist() # Binary prediction where label 1 indicates toxicity.
        prediction_results.append(prediction_list)
        probability_list = torch.softmax(logits, dim=1).detach().numpy()[:,1].tolist() # Probability of toxicity.
        probability_results.append(probability_list)

# Find the indices of the chuncks with HAP prediction of 1
indices = [(i, j) for i, row in enumerate(prediction_results) for j, value in enumerate(row) if value == 1]
# Save the chunks with HAP probability >90% in a key-value dictionary where:
# - key: is the index of the original document
# - value: is a list containg the chunks with HAP content
matrix_of_chunks_with_hap = {}
for tup in indices:
    i=tup[0]; j=tup[1];
    #print(prediction_results[i][j])
    #print(f"Probability: {probability_results[i][j]}, Sentence: {docs_as_list_of_chuncks[i][j]}")
    if(probability_results[i][j] >= 0.90):
        print(tup)  # This prints the tuple
        if(i not in matrix_of_chunks_with_hap):
            matrix_of_chunks_with_hap[i] = []
        matrix_of_chunks_with_hap[i].append(docs_as_list_of_chuncks[i][j])
        print(f"Probability: {probability_results[i][j]}, Sentence: {docs_as_list_of_chuncks[i][j]}")
print(f"There are {len(matrix_of_chunks_with_hap)} documents with at east one sentence with HAP probability > 90%. {list(matrix_of_chunks_with_hap.keys())}.")

/Users/mrinalduzzi/Desktop/Projects/RAG-with-watsonx/HAP/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(0, 109)
Probability: 0.9138151407241821, Sentence: first set in latex and her now-signature white clown face. When it’s time for “Good Luck, Babe!”
(3, 59)
Probability: 0.9971562623977661, Sentence: “We were stupid and didn’t take it seriously. We were not responsible; it was a mistake not to
(19, 26)
Probability: 0.9303997755050659, Sentence: “I’ve used the term hypocrites because we support peaceful protest, and we facilitate that all the
(25, 65)
Probability: 0.9272968173027039, Sentence: sections of the Wall.”
(38, 50)
Probability: 0.9897632598876953, Sentence: You’re a little f**cking confused.”
(38, 135)
Probability: 0.9396336674690247, Sentence: more than that, the thing that f**ks me up, honestly, is knowing that I don’t know exactly what
(40, 19)
Probability: 0.9032238125801086, Sentence: be pouring from her abdomen.
(45, 27)
Probability: 0.9545580148696899, Sentence: “This is scary as sh*t,” the UConn coach conceded.
(61, 25)
Probability: 0.9258887767791748, Sentence: “If yo

In [4]:
import pandas as pd
import requests

from hap_utilities import split_text_preserve_newlines, rebuild_text_with_newlines, find_substring_indices
from os import environ
credentials = {
    "url_LLM": environ.get("WATSONX_URL_LLM"),
    "url_LANGCHAIN": environ.get("WATSONX_URL_LANGCHAIN"),
    "apikey": environ.get("WATSONX_API_KEY")
}
project_id = environ.get("PRJ_ID")

# Generate Bearer token
url = 'https://iam.cloud.ibm.com/identity/token'
myobj = {'grant_type': 'urn:ibm:params:oauth:grant-type:apikey', 'apikey':credentials["apikey"]}
#use the 'headers' parameter to set the HTTP headers:
x = requests.post(url, data = myobj, headers = {"Content-Type": "application/x-www-form-urlencoded"})
bearer_token = x.json()['access_token']


#matrix_of_docs_rewritten = {} # dict (dictionary) that maps integer keys to a single strings (e.g. 38: <document_rewritten>")
for key in list(matrix_of_chunks_with_hap.keys()):
    old_doc = docs_as_list_of_strings[key]
    list_of_hap_strings = matrix_of_chunks_with_hap[key]

    # --- Retrieve only the sentences with HAP content ---
    # Split the text into sentences
    full_strings = split_text_preserve_newlines(old_doc)

    substrings = list_of_hap_strings

    # Find the indices of strings containing any of the substrings
    matching_indices = find_substring_indices(full_strings, substrings)

    #print("Indices of matching strings:", matching_indices)
    # -----------------------------------------------------

    # -----------------------   GRANITE   ------------------------
    # Build the prompt starting from the template and the instructions

    #Ensure that the revised sentences no longer have harmful or offensive language but retain the original meaning or intention in a more respectful and appropriate way.
    for i in range(len(matching_indices)):
        prompt_granite = """Rewrite the following sentence to remove any harmful, abusive, or inappropriate content while maintaining the original meaning where possible. Ensure the revised sentence is respectful, neutral, and appropriate for all audiences. The output should contain only the revised sentence itself, without any additional words, explanations, or punctuation marks.

Input: That was a stupid decision and you’re an idiot for making it.
Output: That was a decision that didn’t work out as planned.

Input: Why are you so incompetent? Can’t you do anything right?
Output: Why are you so ineffective? Can’t you do anything right?

Input: It’s absolutely pathetic how you can’t even get the simplest things right, and it’s clear that you’re just a complete f**king failure in every aspect.
Output: It’s disappointing how you can’t even get the simplest things right, and it’s clear that you need some improvements.


Input: """ + full_strings[matching_indices[i]] + """
Output: """

        body_granite = {
            "input": prompt_granite,
            "parameters": {
                "decoding_method": "greedy",
                "max_new_tokens": 200,
                "repetition_penalty": 1.1
            },
            "model_id": "ibm/granite-13b-instruct-v2",
            "project_id": project_id
        }
        #print(body_granite)

        # Make the API call to the LLM model
        authorization = "Bearer " + bearer_token
        headers = {
            "Accept": "application/json",
            "Content-Type": "application/json",
            "Authorization": authorization
        }

        response = requests.post(
            credentials["url_LLM"],
            headers=headers,
            json=body_granite
        )

        if response.status_code != 200:
            raise Exception("Non-200 response: " + str(response.text))

        result_granite = response.json()["results"][0]["generated_text"]
        # Remove the \n\n at the end of a string if it exists
        if result_granite.endswith('\n\n'):
            result_granite = result_granite[:-2]
        if result_granite.endswith('  '):
            result_granite = result_granite[:-2]
        
        print(matching_indices[i], "original :", full_strings[matching_indices[i]])
        print(matching_indices[i], "rewritten:", result_granite)
        # Change the original sentence with the revised sentence
        full_strings[matching_indices[i]] = result_granite
        
    #matrix_of_docs_rewritten[key] = rebuild_text_with_newlines(full_strings)
    docs[key].page_content = rebuild_text_with_newlines(full_strings)
    docs[key].metadata['rewritten'] = True

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/mrinalduzzi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


32 original : “I’ve used the term hypocrites because we support peaceful protest, and we facilitate that all the time.
32 rewritten: “We call hypocrites because we support peaceful protest, and we facilitate that all the time.”
66 original : Although the couple managed to have a great time traveling independently without knowing Chinese, they decided to hire a private guide for their visit to the Great Wall “as it facilitates logistics and allows you to go to (lesser-traveled) sections of the Wall.”
66 rewritten: Although the couple managed to have a great time traveling independently without knowing Chinese, they decided to hire a private guide for their visit to the Great Wall “since it facilitates logistics and allows you to go to (lesser-traveled) sections of the wall.”
57 original : You’re a little f**cking confused.”
57 rewritten: You're a little confused."
147 original : “And that’s because of my teammates and trying to put myself in that situation that they described emotionall

## Generate embeddings for the extracted articles

Before loading our knowledge base into a suitable vector DB, we need to generate embeddings for our articles.
Embeddings are a type of transformation that helps computers understand the meaning of words and phrases in a text by converting them into a continuous vector space. This makes it easier for computers to learn and make sense of complex relationships between different concepts.

In generative AI pipelines, embeddings are essential because they allow models to capture the semantic meaning of text data. By mapping words and phrases to vectors, embeddings help models maintain context across sentences and documents, making it possible to generate new text that is both coherent and relevant.

To generate embeddings for our articles, we use IBM_SLATE_30M_ENG model from watsonx.ai model library.

N.B. In order to use IBM Slate model within watsonx, you need to set up 3 environment variables, related to your watsonx.ai instance:
- <b>WATSONX_URL</b>: it is a URL with this format "https://{region}.ml.cloud.ibm.com"
- <b>WATSONX_API_KEY</b>: API KEY from your IBM Cloud account. A detailed procedure on how to create an API KEY can be found in the link provided at the end of this cell.
- <b>PRJ_ID</b>: it is the ID of the project created on watsonx.ai platform to run this notebook

Information on how to find/create these variables can be found here: https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-credentials.html?context=wx&audience=wdp.

In [5]:
from langchain_ibm import WatsonxEmbeddings
from ibm_watsonx_ai.foundation_models.utils.enums import EmbeddingTypes
embeddings = WatsonxEmbeddings(
    model_id=EmbeddingTypes.IBM_SLATE_30M_ENG.value,
    #client=client,
    url=credentials["url_LANGCHAIN"],
    apikey=credentials["apikey"],
    project_id=project_id
    )

## Load documents into ChromaDB

With the documents preprocessed and vectorized, we're now ready to load them into ChromaDB. We easily accomplish that leveraging the Chroma integration within Langchain. Once the documents are in Chroma, we can perform a similarity search to retrieve documents related to our topic of interest. Here we choose to limit the retrived documents to 5.

In [6]:
# Split the documents into bigger chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
texts = text_splitter.split_documents(docs)

from langchain_chroma import Chroma
vector_store = Chroma.from_documents(texts, embeddings)
print("Number of documents added to the db:", len(texts))
print("Number of documents within the db:", len(vector_store.get()['documents']))


Number of documents added to the db: 654
Number of documents within the db: 654


We are now ready to receive a question form the user and store it the variable named text

In [7]:
#text = input("Type what you want to search:")
text = "Who is the designer that joined Aston Martin after 19 years with Red Bull?"

We now perform a first similarity search and retrive the 10 most relevant articles and print them out. We expect this search to return documents that are not directly related to the question that the user proposed to the pipeline. We are only sorting the returned documents based on their  distance.

In [8]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1}: \n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )
    
query_docs = vector_store.similarity_search(text, k=10)

pretty_print_docs(query_docs)

Document 1: 

CNN 9/11/2024

‘There’s nobody quite like him in F1’: Legendary designer Adrian Newey joins Aston Martin after 19 years with Red Bull

By Jamie Barton and Amanda Davies, CNN

Updated: 9:09 AM EDT, Tue September 10, 2024

Source: CNN

Aston Martin has won the race to sign Adrian Newey, widely regarded as the greatest Formula One designer of all time.

The 65-year-old has signed a long-term contract with the British team and will join as Managing Technical Partner in March 2025 after nearly 20 years with Red Bull. The deal represents a statement of intent from Aston Martin executive chairman and part-owner Lawrence Stroll.

“I felt as if I needed a new challenge,” Newey said in a press conference at the Aston Martin factory in Silverstone. “I was very flattered to have a lot of approaches from various teams, but Lawrence’s passion, commitment and enthusiasm was very endearing, very persuasive.
---------------------------------------------------------------------------------

## Compress and Summarize the Documents

After retrieving relevant documents from Chroma, we're ready to compress and then summarize them! There are multiple ways to accomplish this in `langchain`, but `ContextualCompressionRetriever` and `load_summarization_chain` is quite straightforward. In this case we're going to use the IBM Granite model within the available Langchain wrapper so to easily integrate it with our summarization chain. Here we limit the summary to snippets related to our topic of choice.

In order to use IBM Granite model within watsonx, we will use the environment variables previously set in the notebook.

In [9]:
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import DecodingMethods
from langchain_ibm import WatsonxLLM

granite = WatsonxLLM(
    model_id='ibm/granite-13b-chat-v2',
    url=credentials["url_LANGCHAIN"],
    apikey=credentials["apikey"],
    project_id=project_id,
    params= {
        GenParams.DECODING_METHOD: DecodingMethods.SAMPLE.value,
        GenParams.MAX_NEW_TOKENS: 1024,
        GenParams.MIN_NEW_TOKENS: 1,
        GenParams.TEMPERATURE: 0.5,
        GenParams.TOP_K: 50,
        GenParams.TOP_P: 1
    }
)

Here we compress the found documents by leveraging the LLM again before passing them into the chain for geneating the answer

In [10]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor   

compressor = LLMChainExtractor.from_llm(granite)
compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=vector_store.as_retriever())

compressed_docs = compression_retriever.invoke(text)
pretty_print_docs(compressed_docs)

Document 1: 

* Adrian Newey, widely regarded as the greatest Formula One designer of all time
* has signed a long-term contract with the British team
* will join as Managing Technical Partner in March 2025
* after nearly 20 years with Red Bull
* The deal represents a statement of intent from Aston Martin executive chairman and part-owner Lawrence Stroll
* Newey said in a press conference at the Aston Martin factory in Silverstone.

No output was generated for the question as the context does not provide any information about the designer joining Aston Martin after 19 years with Red Bull.
----------------------------------------------------------------------------------------------------
Document 2: 

* Adrian Newey, who has joined Aston Martin after 19 years with Red Bull.
* The team currently sits fifth in the constructors’ championship standings, having failed to achieve a podium finish so far this season.
* Neither of Aston Martin’s drivers have made it onto the podium yet this yea

Now we need to set up our summarization chain: the basic chain types are either "stuff" (i.e. documents are provided as context in a single prompt that is passed to the LLM) or "map-reduce" (i.e. documents are processed in a map-reduce fashion in order to obtain summaries from each single document and provide these summaries as context to the LLM). 

In order to avoid possible limits in the number of token processed by the LLM, we could opt for a map-reduce chain (more information on summarization in langchain can be found at https://python.langchain.com/docs/use_cases/summarization) since we applied compression we can stick with the "stuff" type.

We also define a prompt to pass down the invocation along witht he resulting input documents from the retriever and the compressor.

In [11]:
from langchain.chains.summarize import load_summarize_chain
chain = load_summarize_chain(granite, chain_type="stuff")

input = {
    "prompt" : "You are a AI language model designed to function as a specialized Retrieval Augmented Generation (RAG) assistant. When generating responses, prioritize correctness, i.e., ensure that your response is correct given the context and user query, and that it is grounded in the context. Furthermore, make sure that the response is supported by the given document or context. When the question cannot be answered using the context or document, output the following response: 'I'm sorry, I don't know.' Always make sure that your response is relevant to the question. If an explanation is needed, first provide the explanation or reasoning, and then give the final answer.",
    "input_documents" : compressed_docs
}
print(chain.invoke(input)['output_text'])



Adrian Newey, a highly successful Formula One designer, has signed a long-term contract with the British team Aston Martin, leaving his nearly 20-year tenure at Red Bull. This move represents a significant statement of intent from Aston Martin's executive chairman and part-owner Lawrence Stroll. Newey is renowned for his ability to capitalize on changes in regulations, and his arrival at Aston Martin is expected to bolster the team's performance. However, the team is currently fifth in the constructors' championship standings, with no podium finishes so far this season. Ginn, an Australian Olympic athlete, has expressed hope that his Olympic medals, won during a nearly two-decade career, will be returned to him after they were stolen from his car.


Check out IBM _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts.

In [12]:
# ---------------------   CHIAMATA SECCA A WATSONX, SENZA USARE LANGCHAIN   ---------------------

import requests

compressed_docs_as_string = '\n\n'.join(item.page_content for item in compressed_docs)

body = {
	"input": """You are a AI language model designed to function as a specialized Retrieval Augmented Generation (RAG) assistant. When generating responses, prioritize correctness, i.e., ensure that your response is correct given the context and user query, and that it is grounded in the context. Furthermore, make sure that the response is supported by the given document or context. When the question cannot be answered using the context or document, output the following response: '\''I'\''m sorry, I don'\''t know.'\'' Always make sure that your response is relevant to the question. If an explanation is needed, first provide the explanation or reasoning, and then give the final answer.

Input: """ + compressed_docs_as_string + """
Output:""",
	"parameters": {
		"decoding_method": "sample",
		"max_new_tokens": 1024,
		"min_new_tokens": 0,
#		"random_seed": null,
		"stop_sequences": [],
		"temperature": 0.5,
		"top_k": 50,
		"top_p": 1,
		"repetition_penalty": 1
	},
	"model_id": "ibm/granite-13b-chat-v2",
	"project_id": project_id,
	"moderations": {
		"hap": {
			"input": {
				"enabled": True,
				"threshold": 0.5,
				"mask": {
					"remove_entity_value": True
				}
			},
			"output": {
				"enabled": True,
				"threshold": 0.5,
				"mask": {
					"remove_entity_value": True
				}
			}
		}
	}
}

# Generate Bearer token
url = 'https://iam.cloud.ibm.com/identity/token'
myobj = {'grant_type': 'urn:ibm:params:oauth:grant-type:apikey', 'apikey':credentials["apikey"]}
#use the 'headers' parameter to set the HTTP headers:
x = requests.post(url, data = myobj, headers = {"Content-Type": "application/x-www-form-urlencoded"})
bearer_token = x.json()['access_token']

headers = {
	"Accept": "application/json",
	"Content-Type": "application/json",
	"Authorization": "Bearer " + bearer_token
}

response = requests.post(
	url=credentials["url_LLM"],
	headers=headers,
	json=body
)

if response.status_code != 200:
	raise Exception("Non-200 response: " + str(response.text))

data = response.json()

print(data["results"][0]["generated_text"])
print(body)



I'm sorry, I don't know. The context does not provide any information about the designer joining Aston Martin after 19 years with Red Bull.
{'input': "You are a AI language model designed to function as a specialized Retrieval Augmented Generation (RAG) assistant. When generating responses, prioritize correctness, i.e., ensure that your response is correct given the context and user query, and that it is grounded in the context. Furthermore, make sure that the response is supported by the given document or context. When the question cannot be answered using the context or document, output the following response: '''I'''m sorry, I don'''t know.''' Always make sure that your response is relevant to the question. If an explanation is needed, first provide the explanation or reasoning, and then give the final answer.\n\nInput: * Adrian Newey, widely regarded as the greatest Formula One designer of all time\n* has signed a long-term contract with the British team\n* will join as Managing 